# Phase 1: NHANES Data Loading and Harmonization

**Purpose:** Load 45 NHANES XPT files (2007–2012), merge by participant ID within each cycle, then concatenate cycles into a single multi-cycle DataFrame.

**Inputs:** `../data/raw/*.xpt` (45 files: 15 modules × 3 cycles)

**Output:** `../data/processed/01_combined_nhanes.parquet`

**Cycles included:**
- 2007–2008 (suffix `_E`)
- 2009–2010 (suffix `_F`)
- 2011–2012 (suffix `_G`)

**Modules included:** DEMO, BMX, MCQ, SPX, COTNAL, FSQ, HIQ, HSQ, HUQ, PFQ, ECQ, RDQ, SMQ, SMQFAM, SMQRTU

In [1]:
"""
Load all NHANES XPT files for cycles E (2007-08), F (2009-10), G (2011-12).
Merge each cycle's modules on SEQN, then stack cycles vertically.
"""

from pathlib import Path
from functools import reduce
import pandas as pd
import pyreadstat

# --- Paths ---
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Configuration ---
MODULES = [
    "DEMO", "BMX", "MCQ", "SPX", "COTNAL", "FSQ", "HIQ", "HSQ",
    "HUQ", "PFQ", "ECQ", "RDQ", "SMQ", "SMQFAM", "SMQRTU"
]

CYCLES = {
    "E": "2007-2008",
    "F": "2009-2010",
    "G": "2011-2012",
}

# --- Load all files into a nested dict: cycle -> module -> DataFrame ---
print("Loading XPT files...")
print("-" * 60)
data = {}
for suffix, cycle_label in CYCLES.items():
    data[suffix] = {}
    for module in MODULES:
        file_path = RAW_DIR / f"{module}_{suffix}.xpt"
        df, _meta = pyreadstat.read_xport(str(file_path))
        data[suffix][module] = df
        print(f"  {cycle_label}  {module}_{suffix}.xpt:  {df.shape[0]:>6,} rows  ×  {df.shape[1]:>3} cols")
    print()

print("All files loaded.")

Loading XPT files...
------------------------------------------------------------
  2007-2008  DEMO_E.xpt:  10,149 rows  ×   43 cols
  2007-2008  BMX_E.xpt:   9,762 rows  ×   23 cols
  2007-2008  MCQ_E.xpt:   9,666 rows  ×   92 cols
  2007-2008  SPX_E.xpt:   7,743 rows  ×   51 cols
  2007-2008  COTNAL_E.xpt:   8,712 rows  ×    6 cols
  2007-2008  FSQ_E.xpt:  10,149 rows  ×   44 cols
  2007-2008  HIQ_E.xpt:  10,149 rows  ×   17 cols
  2007-2008  HSQ_E.xpt:   9,307 rows  ×   14 cols
  2007-2008  HUQ_E.xpt:  10,149 rows  ×   10 cols
  2007-2008  PFQ_E.xpt:   9,666 rows  ×   55 cols
  2007-2008  ECQ_E.xpt:   3,603 rows  ×   14 cols
  2007-2008  RDQ_E.xpt:   9,666 rows  ×   15 cols
  2007-2008  SMQ_E.xpt:   7,145 rows  ×   37 cols
  2007-2008  SMQFAM_E.xpt:  10,149 rows  ×    5 cols
  2007-2008  SMQRTU_E.xpt:   6,917 rows  ×   24 cols

  2009-2010  DEMO_F.xpt:  10,537 rows  ×   43 cols
  2009-2010  BMX_F.xpt:  10,253 rows  ×   23 cols
  2009-2010  MCQ_F.xpt:  10,109 rows  ×   81 cols
  2009

In [2]:
"""
Within each cycle, merge all 15 modules on SEQN (participant ID).
Use outer joins to retain all participants who appear in any module.
"""

cycle_dfs = {}

for suffix, cycle_label in CYCLES.items():
    print(f"Merging modules for cycle {cycle_label}...")
    module_dfs = list(data[suffix].values())

    # Outer-merge all modules on SEQN
    merged = reduce(
        lambda left, right: pd.merge(left, right, on="SEQN", how="outer"),
        module_dfs
    )

    # Tag with cycle so we can stratify later if needed
    merged["NHANES_CYCLE"] = cycle_label

    cycle_dfs[suffix] = merged
    print(f"  Merged shape: {merged.shape[0]:,} rows × {merged.shape[1]} columns")
    print()

Merging modules for cycle 2007-2008...
  Merged shape: 10,149 rows × 437 columns

Merging modules for cycle 2009-2010...
  Merged shape: 10,537 rows × 401 columns

Merging modules for cycle 2011-2012...
  Merged shape: 9,756 rows × 409 columns



In [3]:
"""
Concatenate the three cycle-level DataFrames into one multi-cycle dataset.
Use outer-join logic: any column present in any cycle is retained;
participants from cycles where that column doesn't exist will have NaN.
"""

combined = pd.concat(
    [cycle_dfs["E"], cycle_dfs["F"], cycle_dfs["G"]],
    axis=0,
    ignore_index=True,
    sort=False,
)

print(f"Combined dataset: {combined.shape[0]:,} rows × {combined.shape[1]} columns")
print()
print("Rows per cycle:")
print(combined["NHANES_CYCLE"].value_counts().sort_index().to_string())
print()
print("First 10 columns:", list(combined.columns[:10]))
print(f"SEQN uniqueness: {combined['SEQN'].is_unique}")

Combined dataset: 30,442 rows × 474 columns

Rows per cycle:
NHANES_CYCLE
2007-2008    10149
2009-2010    10537
2011-2012     9756

First 10 columns: ['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'DMQMILIT']
SEQN uniqueness: True


In [4]:
"""
Save the combined dataset for use in Phase 2 (recoding).
Parquet is faster and smaller than CSV for this kind of mixed-type data.
"""

output_path = PROCESSED_DIR / "01_combined_nhanes.parquet"
combined.to_parquet(output_path, index=False)

size_mb = output_path.stat().st_size / (1024 * 1024)
print(f"Saved: {output_path}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {combined.shape}")

Saved: ..\data\processed\01_combined_nhanes.parquet
Size:  4.0 MB
Shape: (30442, 474)
